# DQ SRC EXCL

Визуализация последнего прогона `dq-src-excl` (логи `DQ_SRC_EXCL`): размеры списков imsi/imei/msisdn.

Перед запуском: `uv run mobile build-src-excl`, затем `uv run mobile dq-src-excl`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_SRC_EXCL"
BOUNDARY = "src_imsi.dataset_presence"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide)
    print("\n--- totals по витринам ---")
    for mart in ("src_imsi", "src_imei", "src_msisdn"):
        row = latest[latest["check"] == f"{mart}.totals"]
        if not row.empty:
            display(row[["check", "status", "metrics"]])

## Визуализации DQ (только метрики из лога)

Все графики — поля `metrics` из JSON-логов `DQ_SRC_EXCL` (без чтения parquet). Прогон в ноутбуке режется по check `src_imsi.dataset_presence`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    render_src_excl_dq_marts,
    render_src_excl_dq_overview,
    src_excl_mart_status_frame,
    src_excl_totals_frame,
)

if latest is None:
    print("Пропуск графиков: нет логов DQ.")
else:
    fig = render_src_excl_dq_overview(latest)
    plt.show()
    fig = render_src_excl_dq_marts(latest)
    plt.show()
    totals = src_excl_totals_frame(latest)
    print(f"Витрин с totals: {len(totals)}")
    if not totals.empty:
        print(totals[["label", "row_count", "unique_count", "null_count", "unique_ratio"]].to_string(index=False))
    print(f"Проверок в матрице статусов: {len(src_excl_mart_status_frame(latest))}")